# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step walkthrough for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is provided according to the [Croissant schema](https://mlcommons.org/croissant/), accessible via a public URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load the dataset metadata and a summary of its structure using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# View summary metadata
meta = dataset.metadata
print(f"\nTitle: {meta.name}\n")
print(f"Description: {meta.description}\n")
print(f"Published: {getattr(meta, 'datePublished', 'Unknown')}")
print(f"License: {meta.license}")


## 2. Data Overview
Let's inspect the available record sets and their structure. All entities in Croissant have unique `@id`s.

We will enumerate the record sets, their IDs, and their fields (columns) as defined in the dataset.

In [ ]:
# List all record sets present in the dataset
print("Available record sets and their fields:")
record_sets = dataset.record_sets

for rs in record_sets:
    print(f"\nRecordSet name: {rs.name}")
    print(f"  RecordSet @id: {rs.id}")
    print(f"  FileObject(s): {[fileobj.id for fileobj in rs.file_objects]}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, column: {field.column.id if field.column else None})")

## 3. Data Extraction
Let's load data from each record set. We'll save each record set's data as a DataFrame, making sure to use the record set `@id` for all references.

In [ ]:
# Assemble all record set @id values
record_set_ids = [rs.id for rs in dataset.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    # Load records as a list of dict (one for each row/record)
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records from RecordSet {record_set_id}.")
    else:
        print(f"No records found for RecordSet {record_set_id}.")

# For demonstration, select the first record set loaded with data
main_record_set_id = next((rs_id for rs_id in record_set_ids if rs_id in dataframes and not dataframes[rs_id].empty), None)

if main_record_set_id:
    print(f"\nRecordSet '{main_record_set_id}' columns:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No populated record sets could be loaded.")

## 4. Exploratory Data Analysis (EDA)
With the main record set loaded, let's explore the data. We'll filter numeric fields, normalize them, and group by a categorical field. All references use the Croissant `@id` for clarity and reproducibility.

In [ ]:
import numpy as np

# Pick a numeric field `@id` from the previously loaded DataFrame
# We'll auto-select the first float/int column for demonstration
numeric_field_id = None
if main_record_set_id:
    df = dataframes[main_record_set_id]
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    
    if numeric_field_id:
        print(f"Selected numeric field for EDA: '{numeric_field_id}'")

        # Example: Filtering out records below a threshold (mean for demonstration)
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered records with '{numeric_field_id}' > {threshold:.2f} (mean):")
        display(filtered_df.head())

        # Normalizing this numeric field
        normed_col = f"{numeric_field_id}_normalized"
        filtered_df[normed_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}' column:")
        display(filtered_df[[numeric_field_id, normed_col]].head())

        # Group by the first non-numeric column for demonstration
        group_field_id = None
        for col in df.columns:
            if not pd.api.types.is_numeric_dtype(df[col]):
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped by '{group_field_id}', mean of '{numeric_field_id}':")
            display(grouped_df.head())
        else:
            print("No non-numeric grouping field available.")
    else:
        print("No numeric field found in the records.")
else:
    print("No data available for EDA.")

## 5. Visualization
Let's plot the distribution of the main numeric field and visualize group means if applicable. This enhances our understanding of the dataset's structure.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[main_record_set_id][numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If grouped_df exists, display barplot
    if 'grouped_df' in locals() and group_field_id:
        plt.figure(figsize=(10,4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
        plt.title(f"Mean '{numeric_field_id}' by '{group_field_id}'")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
- This notebook demonstrated loading and basic analysis of a Croissant-formatted FAIR dataset using `mlcroissant`.
- We explored record set structures, loaded and processed a record set using only Croissant `@id` references, and conducted basic EDA including visualization.
- For deeper analysis, consult the detailed field and record set documentation accessible via the Croissant schema.